# FG-NET complementary-data analysis

This notebook does not download data. Point `FGNET_IMAGES_ROOT` to the flat folder containing files such as `001A02.JPG`. It audits all valid longitudinal pairs and previews the scarcity-aware subset added without removing Colombian observations.

In [ ]:
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from data import (
    build_fgnet_manifest,
    build_pair_index,
    transition_cell,
)

In [ ]:
FGNET_IMAGES_ROOT = Path('/server/path/to/FGNET/images')
COLOMBIAN_DATASET_ROOT = Path('/server/path/to/longitudinal_faces')

## 1. Manifest and exact pair inventory

In [ ]:
fgnet_root, fgnet_manifest, fgnet_audit = build_fgnet_manifest(FGNET_IMAGES_ROOT)
fgnet_pairs = build_pair_index(fgnet_manifest, min_age_gap=1)

print(f"People: {fgnet_audit['identities']:,}")
print(f"Images: {fgnet_audit['images']:,}")
print(f"All eligible forward image-pairs: {len(fgnet_pairs):,}")
print(f"Age range: {min(row.age for row in fgnet_manifest)}-{max(row.age for row in fgnet_manifest)}")

images_df = pd.DataFrame({
    'person_id': [row.person_id for row in fgnet_manifest],
    'age': [row.age for row in fgnet_manifest],
    'filename': [row.filename for row in fgnet_manifest],
})
pairs_df = pd.DataFrame({
    'person_id': [pair.person_id for pair in fgnet_pairs],
    'source_age': [pair.source_age for pair in fgnet_pairs],
    'target_age': [pair.target_age for pair in fgnet_pairs],
    'delta_age': [pair.delta_age for pair in fgnet_pairs],
})
display(images_df.describe(include='all'))
display(pairs_df.describe(include='all'))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
images_df.age.hist(bins=range(0, 72, 2), ax=axes[0])
axes[0].set(title='FG-NET images by age', xlabel='Age', ylabel='Images')
pairs_df.delta_age.hist(bins=range(1, 71, 2), ax=axes[1])
axes[1].set(title='All forward-pair gaps', xlabel='Age gap', ylabel='Pairs')
pairs_df.plot.hexbin(x='source_age', y='target_age', gridsize=20, ax=axes[2], cmap='viridis')
axes[2].set_title('All source → target transitions')
plt.tight_layout()

## 2. Preview the production selection

The loader compares source/target/gap cells against Colombian coverage, fills the least represented cells first, and balances FG-NET identities. `KAGGLE_PROPORTION=0.40` adds `0.40 × Colombian epoch observations`; it never removes Colombian data. The special value `1.0` selects every eligible FG-NET pair.

In [ ]:
from data import build_face_aging_dataloaders

KAGGLE_PROPORTION = 0.40
loaders, metadata = build_face_aging_dataloaders(
    COLOMBIAN_DATASET_ROOT,
    image_size=256, batch_size=4, num_workers=0, train_drop_last=False,
    include_bidirectional_pairs=True, reverse_pair_prob=0.20,
    include_kaggle=True, kaggle_path=FGNET_IMAGES_ROOT,
    kaggle_proportion=KAGGLE_PROPORTION,
    kaggle_reverse_pair_prob=0.50,
)
display(pd.Series(metadata['kaggle'], name='FG-NET integration'))
selected = loaders['train'].dataset.complementary_pairs
selected_df = pd.DataFrame({
    'person_id': [pair.person_id for pair in selected],
    'source_age': [pair.source_age for pair in selected],
    'target_age': [pair.target_age for pair in selected],
    'delta_age': [pair.delta_age for pair in selected],
    'transition_cell': [' | '.join(transition_cell(pair)) for pair in selected],
})
display(selected_df.transition_cell.value_counts().head(30))

## 3. Verify rejuvenation exposure

FG-NET defaults to 50% reverse ordering when bidirectional training is enabled. This is measured across epochs because reversal is deterministic per seed, epoch, and index.

In [ ]:
fgnet_dataset = loaders['train'].dataset.complementary
directions = Counter()
for epoch in range(20):
    fgnet_dataset.set_epoch(epoch)
    for index in range(len(fgnet_dataset)):
        delta = fgnet_dataset.pair_for_index(index).delta_age
        directions['reverse' if delta < 0 else 'forward'] += 1
total = sum(directions.values())
print(directions)
print({key: value / total for key, value in directions.items()})